In [1]:
import numpy as np
import torch
from timestral import TimestralForecaster

model = TimestralForecaster.from_pretrained(
    "./models/Timestral-209M",
    device="cuda" if torch.cuda.is_available() else "cpu",
    local_files_only=True
)

In [3]:
# Synthetic example: one series with 1,000 historical observations.
context = np.random.default_rng(42).standard_normal(1000).cumsum()

forecast = model.forecast(context, prediction_length=96)

median = forecast.median
lower = forecast.quantile(0.1)
upper = forecast.quantile(0.9)
print(median)

tensor([[[-28.8507, -28.8312, -28.8243, -28.8372, -28.8261, -28.8107, -28.8137,
          -28.8284, -28.8333, -28.8155, -28.8377, -28.8590, -28.8470, -28.8957,
          -28.8673, -28.8931, -28.8808, -28.8820, -28.8987, -28.9048, -28.9220,
          -28.9105, -28.8976, -28.9066, -28.9166, -28.9182, -28.9209, -28.9319,
          -28.9143, -28.9299, -28.9279, -28.9051, -28.9218, -28.9157, -28.9086,
          -28.8981, -28.8876, -28.8708, -28.8607, -28.8656, -28.8931, -28.8645,
          -28.8587, -28.8486, -28.8307, -28.8703, -28.8515, -28.8642, -28.8423,
          -28.8441, -28.8548, -28.8499, -28.8626, -28.8500, -28.8495, -28.8404,
          -28.8436, -28.8528, -28.8435, -28.8485, -28.8267, -28.8220, -28.7905,
          -28.7815, -28.7847, -28.7891, -28.7954, -28.7810, -28.7610, -28.7299,
          -28.7026, -28.7187, -28.7528, -28.7221, -28.7168, -28.7235, -28.7175,
          -28.7511, -28.7451, -28.7696, -28.7512, -28.7536, -28.7589, -28.7604,
          -28.7658, -28.7615, -28.7584, 

In [ ]:
# later testing

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from timestral import TimestralForecaster

# 1. Read the real dataset
data = pd.read_csv("datasets/ETTh1.csv", parse_dates=["date"])
data = data.sort_values("date")

# 2. Select the oil temperature column
values = data["OT"].to_numpy(dtype=np.float32)

# 3. Keep the last 96 hours for testing
context = values[-1096:-96]
actual = values[-96:]
dates = data["date"].iloc[-96:]

# 4. Load your locally downloaded model
model = TimestralForecaster.from_pretrained(
    "./models/Timestral-209M",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 5. Predict the next 96 hours
forecast = model.forecast(context, prediction_length=96)

predicted = np.asarray(forecast.median).reshape(-1)
lower = np.asarray(forecast.quantile(0.1)).reshape(-1)
upper = np.asarray(forecast.quantile(0.9)).reshape(-1)

# 6. Measure prediction error
mae = np.mean(np.abs(predicted - actual))
print(f"Mean Absolute Error: {mae:.2f}")

# 7. Plot the results
plt.figure(figsize=(12, 5))
plt.plot(dates, actual, label="Actual Temperature")
plt.plot(dates, predicted, label="Timestral Prediction")
plt.fill_between(dates, lower, upper, alpha=0.2, label="10-90% Forecast Range")

plt.title("Timestral vs Actual Transformer Temperature")
plt.xlabel("Date")
plt.ylabel("Oil Temperature")
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
